In [14]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
import pandas as pd
from src.utils import build_data_from_suffix
from src.utils import save_str_ud_deprel_mismatches

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
DATA = build_data_from_suffix("syntax", csv_dir="aligned")

# Анализ ошибок: дуплеты и триплеты

In [ ]:
def get_ud_errors_with_str(data, split):
    """Return all rows where UD prediction is wrong, merged with STR data."""
    str_key = "str"  if split == "full" else f"str-{split}"
    ud_key  = "ud"   if split == "full" else f"ud-{split}"
    str_df = data[str_key].reset_index()
    ud_df  = data[ud_key].reset_index()
    merged = str_df.merge(ud_df, on=["sent_id", "id"], suffixes=("_str", "_ud"))
    return merged[merged["deprel_g_ud"] != merged["deprel_p_ud"]].copy()


def top_per_split(data, group_cols, params, n=10, noisy_only=False):
    """
    Build a rank table: rows = rank 1..n, columns = full / old / new.
    noisy_only=True keeps only rows where STR prediction is also wrong.
    """
    split_tops = {}
    for split in ("full", "old", "new"):
        errors = get_ud_errors_with_str(data, split)
        if noisy_only:
            errors = errors[errors["deprel_g_str"] != errors["deprel_p_str"]]
        top = (
            errors
            .groupby(group_cols)
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
            .head(n)
            .reset_index(drop=True)
        )
        split_tops[split] = top

    rows = []
    for i in range(n):
        row = {"#": i + 1}
        for split in ("full", "old", "new"):
            top = split_tops[split]
            for c in params:
                row[f"{split}_{c}"] = top.iloc[i][c] if i < len(top) else ""
        rows.append(row)

    return pd.DataFrame(rows).set_index("#")

## Таблица 1 — Топ-10 дуплетов (UD_gold, UD_predicted) по всем UD-ошибкам

In [34]:
duplets = top_per_split(
    DATA,
    group_cols=["deprel_g_ud", "deprel_p_ud"],
    params=["deprel_g_ud", "deprel_p_ud", "count"],
    n=10,
)
duplets.to_csv("matrix_duplets.csv")
display(duplets)

,full_deprel_g_ud,full_deprel_p_ud,full_count,old_deprel_g_ud,old_deprel_p_ud,old_count,new_deprel_g_ud,new_deprel_p_ud,new_count
#,,,,,,,,,
1,nmod,obl,393,nmod,obl,306,nmod,obl,89
2,obl,nmod,367,obl,nmod,275,obl,nmod,80
3,obj,obl,159,nsubj:pass,nsubj,78,nsubj:pass,nsubj,30
4,iobj,obl,158,appos,parataxis,75,root,conj,25
5,xcomp,obl,144,obl,obj,69,nsubj,obj,23
6,obl,obj,123,nsubj,nsubj:pass,64,parataxis,conj,23
7,appos,parataxis,114,nmod,appos,61,parataxis,root,22
8,nsubj:pass,nsubj,100,root,nsubj,60,root,nsubj,21
9,case,mark,94,conj,parataxis,59,iobj,obl:agent,20


## Таблица 2 — Топ-10 триплетов (STR_predicted≠gold, UD_gold, UD_predicted) — только noisy-случаи

В шумном случае (STR тоже предсказан неверно) добавление неправильного STR в качестве третьей координаты сильно дробит группы: топ-10 триплетов покрывают гораздо меньшую долю ошибок, чем топ-10 дуплетов.

In [35]:
triplets = top_per_split(
    DATA,
    group_cols=["deprel_p_str", "deprel_g_ud", "deprel_p_ud"],
    params = ["deprel_p_str", "deprel_g_ud", "deprel_p_ud", "count"],
    n=10,
    noisy_only=True,
)
triplets.to_csv("matrix_triplets.csv")
display(triplets)

,full_deprel_p_str,full_deprel_g_ud,full_deprel_p_ud,full_count,old_deprel_p_str,old_deprel_g_ud,old_deprel_p_ud,old_count,new_deprel_p_str,new_deprel_g_ud,new_deprel_p_ud,new_count
#,,,,,,,,,,,,
1,обст,nmod,obl,174,атриб,obl,nmod,139,обст,nmod,obl,43
2,атриб,obl,nmod,156,обст,nmod,obl,135,атриб,obl,nmod,38
3,аппоз,nmod,appos,48,аппоз,nmod,appos,45,1-компл,nsubj,obj,18
4,root,nsubj,root,43,предик,obj,nsubj,35,предик,obj,nsubj,14
5,предик,root,nsubj,40,предик,root,nsubj,35,root,parataxis,root,13
6,предик,obj,nsubj,39,root,nsubj,root,33,предик,root,nsubj,12
7,разъяснит,appos,parataxis,36,2-компл,nmod,obl,31,количест,det,nummod,10
8,2-компл,nmod,obl,35,разъяснит,appos,parataxis,26,сент-соч,root,conj,9
9,1-компл,obl,nmod,35,1-компл,nsubj,obj,26,1-компл,iobj,obj,9


In [ ]:
matrix = pd.read_csv("duplets/matrix_duplets.csv")
for i in range(10):
    row = matrix.iloc[i]
    for split in ("full", "old", "new"):
        deprel_g = row[f"{split}_deprel_g_ud"]
        deprel_p = row[f"{split}_deprel_p_ud"]
        save_str_ud_deprel_mismatches(
            DATA,
            split=split,
            deprel_str="any",
            deprel_ud_gold=deprel_g,
            deprel_ud_predicted=deprel_p,
        )

In [41]:
from src.utils import save_noisy_triplet

matrix = pd.read_csv("triplets/matrix_triplets.csv")
for i in range(10):
    row = matrix.iloc[i]
    for split in ("full", "old", "new"):
        deprel_str = row[f"{split}_deprel_p_str"]
        deprel_g = row[f"{split}_deprel_g_ud"]
        deprel_p = row[f"{split}_deprel_p_ud"]
        out = save_noisy_triplet(DATA, split, deprel_str, deprel_g, deprel_p)
